# nanoGPT — Arabic Language Model

**Pipeline:** Arabic Text → Tokenization (Aranizer-SP-32k) → Dataset Shards (90/10) → Training → Loss Plot

**Dataset:** Arabic Wikipedia (HuggingFace `wikipedia 20220301.ar`)  
**Tokenizer:** `riotu-lab/Aranizer-SP-32k` (SentencePiece, vocab=32k)  
**Model:** nanoGPT (Karpathy)  




## Step 1: Install Dependencies

In [ ]:
!pip install torch numpy transformers datasets tiktoken tqdm sentencepiece -q
print('Done installing')

## Step 2: Clone nanoGPT

In [ ]:
import os

if not os.path.exists('nanoGPT'):
    !git clone https://github.com/karpathy/nanoGPT.git

%cd nanoGPT
print('nanoGPT ready')

## Step 3: Load Aranizer-SP-32k Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('riotu-lab/Aranizer-SP-32k')

sample = 'مرحباً بالعالم، هذا اختبار للمجزئ العربي.'
tokens = tokenizer.encode(sample)
decoded = tokenizer.decode(tokens)

print(f'Vocab size : {tokenizer.vocab_size:,}')
print(f'Sample     : {sample}')
print(f'Token IDs  : {tokens[:8]}...')
print(f'Decoded    : {decoded}')

## Step 4: Load Arabic Wikipedia Dataset



In [ ]:
from datasets import load_dataset

MAX_ARTICLES = 50000

dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.ar",
    split="train"
)

dataset = dataset.select(range(min(MAX_ARTICLES, len(dataset))))

print(f'Articles loaded : {len(dataset):,}')
print(f'Sample title    : {dataset[0]["title"]}')
print(f'Sample text     : {dataset[0]["text"][:200]}')

## Step 5: Tokenize & Create Shards (90% train / 10% val)

In [ ]:
import numpy as np
import os
from tqdm import tqdm

DATA_DIR = 'data/arabic_wiki'
os.makedirs(DATA_DIR, exist_ok=True)

all_ids = []
eos_id = tokenizer.eos_token_id or 1

for article in tqdm(dataset, desc='Tokenizing'):
    text = article['text'].strip()
    if not text:
        continue
    ids = tokenizer.encode(text, add_special_tokens=False)
    ids.append(eos_id)
    all_ids.extend(ids)

print(f'Total tokens: {len(all_ids):,}')

split = int(len(all_ids) * 0.9)
train_ids = np.array(all_ids[:split], dtype=np.uint16)
val_ids   = np.array(all_ids[split:], dtype=np.uint16)

train_ids.tofile(os.path.join(DATA_DIR, 'train.bin'))
val_ids.tofile(  os.path.join(DATA_DIR, 'val.bin'))

print(f'Train: {len(train_ids):,} tokens ({train_ids.nbytes/1e6:.1f} MB)')
print(f'Val  : {len(val_ids):,} tokens ({val_ids.nbytes/1e6:.1f} MB)')

## Step 6: Write Training Config

In [ ]:
config = '''
# nanoGPT config — Arabic Wikipedia + Aranizer-SP-32k
dataset          = 'arabic_wiki'
out_dir          = 'out-arabic'

# Model (Baby GPT — fits T4 16GB)
n_layer          = 6
n_head           = 6
n_embd           = 384
block_size       = 128
dropout          = 0.1
vocab_size       = 32000

# Training
batch_size       = 16
max_iters        = 500
learning_rate    = 1e-3
lr_decay_iters   = 500
min_lr           = 1e-4
warmup_iters     = 100
beta2            = 0.99

# Eval & Logging
eval_interval    = 50
eval_iters       = 20
log_interval     = 10

# Hardware
device           = 'cuda'
compile          = False
dtype            = 'float16'
always_save_checkpoint = True
'''

with open('config/train_arabic.py', 'w', encoding='utf-8') as f:
    f.write(config)

print('Config saved!')
print(config)

## Step 7: Verify GPU & Files

In [ ]:
import torch, os

print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

for f in ['data/arabic_wiki/train.bin', 'data/arabic_wiki/val.bin']:
    size = os.path.getsize(f)/1e6
    print(f'  {f}: {size:.1f} MB')

## Step 8: Train



In [ ]:
!python train.py config/train_arabic.py 2>&1 | tee training_log.txt

## Step 9: Plot Loss

In [ ]:
import re, matplotlib.pyplot as plt

val_steps, train_losses, val_losses = [], [], []

with open('training_log.txt') as f:
    for line in f:
        m = re.search(r'iter (\d+):.*train loss ([\d.]+).*val loss ([\d.]+)', line)
        if m:
            val_steps.append(int(m.group(1)))
            train_losses.append(float(m.group(2)))
            val_losses.append(float(m.group(3)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(val_steps, train_losses, label='Train Loss', color='#2196F3', linewidth=2)
ax.plot(val_steps, val_losses,   label='Val Loss',   color='#F44336', linewidth=2, linestyle='--')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('nanoGPT — Arabic Wikipedia (Aranizer-SP-32k)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_plot.png', dpi=150)
plt.show()

if val_losses:
    print(f'Best val loss  : {min(val_losses):.4f}')
    print(f'Final val loss : {val_losses[-1]:.4f}')

## Step 10: Generate Arabic Text

In [ ]:
import torch, sys
sys.path.insert(0, '.')
from model import GPTConfig, GPT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('out-arabic/ckpt.pt', map_location=device)
model = GPT(GPTConfig(**ckpt['model_args']))
model.load_state_dict(ckpt['model'])
model.eval().to(device)

prompt = 'اللغة العربية هي'
ids = tokenizer.encode(prompt, add_special_tokens=False)
x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

with torch.no_grad():
    y = model.generate(x, max_new_tokens=100, temperature=0.8, top_k=50)

print(tokenizer.decode(y[0].tolist()))

## (Optional) Download Outputs

In [ ]:
from google.colab import files
files.download('loss_plot.png')
# files.download('out-arabic/ckpt.pt')  # uncomment لتحميل الموديل